**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**01 | Source and Reference Eligibility**

![Source and Reference Eligibility](../../images/01_source_and_reference_eligibility.svg)

**Objective:** Inspect original IMU and treadmill force-platform recordings, identify same-trial measurement pairs, and establish the requirements for force-referenced gait-event evaluation.

**Dataset:** Camargo et al. (2021), original treadmill sensor streams. The last executed source audit used `AB06/10_09_18/treadmill_01_01`; the revised acquisition stage proposes a nine-person AB06–AB14 source sample for Colab Free. This expanded sample has **not yet been executed**.

**Notebook scope:** Original-source inventory, MATLAB channel and `Header` inspection, recorded signal preview and eligibility status. No force-derived contact labels, fitted models or event-detection accuracy are produced in this stage.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

Initialise the shared Colab environment, the MATLAB table reader, scientific utilities and study output directory. Section 02 reuses an existing source pair or automatically downloads and extracts the original one-participant pilot when local data are absent.


In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 1

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib.util
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import os
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import (
    inventory_files, list_archives, paired_trials,
    read_sensor_table, validate_timebase, inspect_columns,
    extract_pilot_trial_from_zip, locate_pilot_source_archive,
    download_mendeley_pilot_archive, PART1_V1_SUBJECT_ARCHIVES,
    extract_selected_trial_pairs_from_zip, participant_pair_counts,
)
from gait_alignment import candidate_channels, preview_channels

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get(
    "CAMARGO_DATA_ROOT", root / "data/raw/camargo"
))
output_dir = study_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
print("Original source root:", source_root)
print("Scientific output folder:", output_dir)


Repository up to date.
Original source root: /content/human-movement-intelligence/data/raw/camargo
Scientific output folder: /content/human-movement-intelligence/gait_analysis/output


**02. Prepare a Resource-Bounded Camargo Source Cohort**

The original Camargo Part 1 Version 1 contains **nine individually downloadable participant archives, AB06–AB14**. This notebook is configured for a bounded **nine-person source sample**, not automatic acquisition of all 22 participants. Original source availability and eligibility must still be confirmed after inspection.

**Colab Free strategy:** acquire at most **three new participants per execution of this cell**, downloading **one ZIP at a time**, extracting up to **three treadmill IMU/force trial pairs per person**, and deleting only a ZIP that this execution downloaded successfully. Existing original source files are reused. The full nine-person sample therefore normally requires repeated execution of **Section 02** in the same runtime, followed by rerunning Sections 03–07 to refresh saved evidence. An interrupted download can be resumed where the publisher supports it. The total network transfer may still be substantial (approximately 9 GB for the nine published archives); this workflow controls **simultaneous disk use**, not total bytes downloaded.

The three trials are selected deterministically across sorted available original trial identifiers. This is **not** a verified sample of particular treadmill speeds. The extracted participant count is a *source-pairing count*, not yet the eligible modelling cohort. Notebook 02 must independently review the force axis, side, units, timing and contact reference before assigning participant-disjoint evaluation roles. Data in Colab's temporary runtime may disappear after disconnecting, so keep downloaded scientific output ZIPs; original data are not stored in GitHub.


In [ ]:
# ================ 02. Prepare Resource-Bounded Camargo Cohort ================

# ---------------- 1. Scientific Sampling and Resource Limits -----------------
# Public Part 1 Version 1 supplies AB06–AB14 as individual ZIP files.
TARGET_SUBJECTS = tuple(name.removesuffix(".zip")
                        for name in PART1_V1_SUBJECT_ARCHIVES)
MAX_TRIAL_PAIRS_PER_PERSON = 3
MAX_NEW_SUBJECTS_PER_RUN = 3
AUTO_DOWNLOAD_PILOT = True
DELETE_NEW_ARCHIVE_AFTER_EXTRACTION = True

# Optional: existing AB06 ZIP in Google Drive, otherwise no Drive needed.
MOUNT_GOOGLE_DRIVE = False
PILOT_ZIP_PATH = None
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


# --------------- 2. Check Original Data and Remaining Subjects ---------------
import shutil

source_root.mkdir(parents=True, exist_ok=True)
counts = participant_pair_counts(source_root, TARGET_SUBJECTS)
pending = [name for name in TARGET_SUBJECTS
           if counts[name] < MAX_TRIAL_PAIRS_PER_PERSON]
print("Current paired-trial coverage:", counts)
print("Participants requiring selected trials:", pending)


# ------------- 3. Acquire and Extract a Small Participant Batch --------------
for subject_id in pending[:MAX_NEW_SUBJECTS_PER_RUN]:
    print("Preparing original participant:", subject_id)
    selected_zip = locate_pilot_source_archive(
        source_root, subject_id=subject_id,
        explicit_archive=(PILOT_ZIP_PATH if subject_id == "AB06" else None),
    )
    downloaded_now = False
    if selected_zip is None:
        if not AUTO_DOWNLOAD_PILOT:
            raise FileNotFoundError(
                f"No source archive for {subject_id}; enable automatic "
                "acquisition or provide the original ZIP."
            )
        free_bytes = shutil.disk_usage(source_root).free
        if free_bytes < 2_200_000_000:
            raise OSError(
                f"Insufficient Colab disk space for a single participant: "
                f"{free_bytes / 1e9:.2f} GB free. Restore available storage "
                "before downloading; raw archives will not be committed."
            )
        selected_zip = download_mendeley_pilot_archive(
            source_root, filename=f"{subject_id}.zip",
            dataset_id="fcgm3chfff", version=1,
        )
        downloaded_now = True
    selected = extract_selected_trial_pairs_from_zip(
        selected_zip, source_root, subject_id=subject_id,
        condition="treadmill",
        max_pairs=MAX_TRIAL_PAIRS_PER_PERSON,
    )
    print("Selected original paired treadmill trials:", len(selected))
    verified = participant_pair_counts(source_root, [subject_id])
    if verified[subject_id] < len(selected):
        raise ValueError(
            f"Incomplete original IMU/force extraction for {subject_id}."
        )
    if downloaded_now and DELETE_NEW_ARCHIVE_AFTER_EXTRACTION:
        selected_zip.unlink()
        print("Removed completed temporary ZIP to conserve Colab storage.")


# ---------------- 4. Report Participant-Level Source Coverage ----------------
coverage = participant_pair_counts(source_root, TARGET_SUBJECTS)
covered = sum(count > 0 for count in coverage.values())
fully_sampled = sum(count >= MAX_TRIAL_PAIRS_PER_PERSON
                    for count in coverage.values())
print("Source-paired participants:", covered, "of", len(TARGET_SUBJECTS))
print("Participants with selected three-trial sample:", fully_sampled)
print("Coverage by participant:", coverage)
if fully_sampled < len(TARGET_SUBJECTS):
    print(
        "Source acquisition is incomplete. Rerun this Section 02 in the "
        "same Colab runtime to collect the next bounded batch, then run "
        "Sections 03–07 again to refresh current research outputs."
    )
else:
    print(
        "All selected source pairs are present. Continue with the "
        "source audit; scientific reference eligibility remains unverified."
    )


**03. Inspect Original Archive and Sensor Inventory**

Inventory the original files **currently present** in the Colab runtime. Previously executed Notebook 01 evidence includes one IMU file and one force-platform file for AB06. After the revised bounded acquisition stage is run to completion, this inventory will include the additional selected participants and treadmill trials. Inspect actual participant coverage rather than treating the nine-person target as an executed count.


In [ ]:
# ======================= 03. Original Source Inventory =======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
if not source_root.exists():
    raise FileNotFoundError(
        f"No Camargo source found at {source_root}. "
        "Download/extract one participant (ABxx) from the official Georgia Tech "
        "Camargo dataset, or set CAMARGO_DATA_ROOT to the mounted data folder."
    )
archives = list_archives(source_root)
inventory = inventory_files(source_root)
display(archives.head(10), inventory.head(15))
print("Observed subjects:", inventory.subject_id.nunique())
print("Observed sensor records:", inventory.sensor.value_counts().to_dict())

# ------------------------- 2. Save Scientific Tables -------------------------
inventory.to_csv(output_dir / "01_trial_inventory.csv", index=False)
if inventory.empty:
    raise ValueError(
        "No extracted ABxx/.../imu or fp source records were discovered. "
        "ZIP files alone are not decoded; inspect archive structure and "
        "extract one participant before proceeding."
    )


**04. Identify Candidate Paired Trials**

Match each original IMU file with its force-platform counterpart using participant, session, locomotor condition and trial identifier. The previous executed AB06 audit identified one candidate same-trial pair; the updated source inventory will be larger **only when new participants have actually been downloaded and extracted**.

Source identity matching is not proof of physical clock synchronisation, anatomical foot correspondence, or a valid force-contact reference.


In [ ]:
# ===================== 04. Identify Paired Sensor Trials =====================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
pairs = paired_trials(inventory)
display(pairs.head(15))
print("Candidate IMU/FP pairs:", int(pairs.candidate_pair.sum()))
pairs.to_csv(output_dir / "01_candidate_pairs.csv", index=False)
if not pairs.candidate_pair.any():
    raise ValueError(
        "No same-trial IMU/force pairs were detected. Inspect source paths "
        "and exact trial-name conventions before extending the adapter."
    )


**05. Inspect Original Trial Headers and Recorded Channels**

Read both original MATLAB tables and audit their numeric source columns. The original AB06 tables were decoded successfully, including their `Header` columns, without manually exporting the files to CSV.

The [observed-column inventory](../../output/01_observed_columns.csv) contains **24 IMU measurement fields** and **18 force-platform fields**, excluding their respective `Header` columns. The recorded median `Header` increments are **0.005** for IMU and **0.001** for the force-platform table, in source-time units. These observations do not independently verify the physical units or timebase synchronisation.


In [ ]:
# ========================= 05. Inspect Trial Headers =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
sample = pairs.loc[pairs.candidate_pair].iloc[0]
imu_file = source_root / sample.imu_path
fp_file = source_root / sample.fp_path
imu_table = read_sensor_table(imu_file)
fp_table = read_sensor_table(fp_file)
imu_time = validate_timebase(imu_table, "IMU")
fp_time = validate_timebase(fp_table, "Force")
column_audit = pd.concat([
    inspect_columns(imu_table, imu_file).assign(modality="imu"),
    inspect_columns(fp_table, fp_file).assign(modality="fp"),
], ignore_index=True)
display(column_audit)

# ------------------------- 2. Save Scientific Tables -------------------------
column_audit.to_csv(output_dir / "01_observed_columns.csv", index=False)
print("IMU likely gyro/segment field names:", candidate_channels(imu_table, "imu"))
print("Potential force field names:", candidate_channels(fp_table, "fp"))
print("Native IMU time step (source units):", np.median(np.diff(imu_time)))
print("Native Force time step (source units):", np.median(np.diff(fp_time)))


**06. Visualise Recorded IMU and Treadmill Source Channels**

The [saved signal preview](../../output/01_sensor_source_preview.png) displays approximately five `Header` time units from the original AB06 trial. The IMU panel shows `foot_Gyro_Y` and `shank_Gyro_Y`; the treadmill panel shows `Treadmill_R_vy` and `Treadmill_L_vy`. The two treadmill components exhibit alternating changes in their recorded values.

Both panels use the original source field names and retain unverified source units. Their appearance alone cannot establish the physical vertical-force convention, force sign, anatomical foot mapping or cross-device synchronisation. No Initial Contact (IC) or Terminal Contact (TC) labels are inferred in this source-inspection stage.


In [ ]:
# ======================= 06. Visualise Source Signals ========================

# ------------------ 1. Choose Observed Source Channel Names ------------------
imu_channels = preview_channels(imu_table, "imu")
force_channels = preview_channels(fp_table, "fp")
if not imu_channels or not force_channels:
    raise ValueError("Insufficient numeric sensor columns for preview.")
print("IMU gyroscope channels:", imu_channels)
print("Treadmill force-platform components:", force_channels)
print("These plotted channels are not yet validated contact references.")

# --------------- 2. Plot Short Recorded Native-Time Intervals ----------------
fig, axes = plt.subplots(
    2, 1, figsize=(11, 6.4), sharex=False, layout="constrained"
)
panels = (
    (axes[0], imu_table, imu_time, imu_channels,
     "Observed IMU gyroscope", ("#236CDE", "#168C5B")),
    (axes[1], fp_table, fp_time, force_channels,
     "Observed treadmill force-platform components",
     ("#E87929", "#236CDE")),
)
for ax, table, time, columns, title, colours in panels:
    native_time = np.asarray(time, dtype=float)
    view = native_time <= native_time[0] + 5.0
    if np.count_nonzero(view) < 3:
        raise ValueError("Too few native-time samples for signal preview.")
    for column, colour in zip(columns[:2], colours):
        signal = pd.to_numeric(table.loc[view, column], errors="coerce")
        ax.plot(native_time[view], signal, color=colour,
                linewidth=1.0, label=column)
    ax.set_title(title, loc="left")
    ax.set_ylabel("Channel value (source units)")
    ax.set_xlabel("Stored Header (source time units)")
    ax.legend(loc="upper right", ncol=2, frameon=False)
    ax.grid(alpha=0.2)
fig.suptitle(
    f"{sample.subject_id} / {sample.trial_id}: recorded sensor signals"
)

# --------------------- 3. Export Observed Signal Preview ---------------------
save_figure(fig, output_dir / "01_sensor_source_preview.png")
plt.show()


**07. Research Summary and Output Download**

The existing published source-audit evidence covers **AB06 and one paired treadmill trial**. The revised Section 02 supports a bounded nine-person AB06–AB14 acquisition, in batches, without requiring all original ZIP files to remain on the Colab disk together. The summary values below will reflect **what was actually prepared during the present runtime**, not the planned target.

The five stage-owned `01_` outputs are updated on execution, then packaged by the shared output helper. Retrieve the ZIP from Colab. Source archives and MATLAB recordings are not included.

**Next scientific dependency:** Notebook 02 must verify the selected foot/shank gyroscope channel, vertical-force orientation and sign, matching foot, physical timestamp units and IMU/force alignment **before** calculating force-derived IC/TC events. More source-paired participants improve feasibility but do not automatically establish reference eligibility. The nine-person cohort offers a limited participant-independent pilot; the final Test will involve few people and uncertainty must be interpreted accordingly.


In [ ]:
# ================= 07. Research Summary and Output Download ==================

# ------------------- 1. Summarise Original Source Evidence -------------------
summary = pd.DataFrame([
    {"item": "source_participants_observed",
     "value": int(inventory.subject_id.nunique())},
    {"item": "candidate_imu_force_pairs",
     "value": int(pairs.candidate_pair.sum())},
    {"item": "validated_reference_pairs",
     "value": "pending original-time/channel/side inspection"},
    {"item": "force_contact_event_results",
     "value": "not generated in source inspection"},
])
display(summary)

# ------------------------ 2. Save Eligibility Summary ------------------------
summary.to_csv(output_dir / "01_source_eligibility_summary.csv", index=False)

# -------------------- 3. Package Current Notebook Outputs --------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
